### Create transaction before doing anything first :3

In [ ]:
from utils.preprocess import create_transactions
from utils.postprocess import map_movie_id_to_title
import pandas as pd
# transaction_list = create_transactions_from_ratings(r'/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/rating.csv', save_path='/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/transaction.pkl')
transaction_list = create_transactions(r'/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_1m/ratings.csv',
                                       chunk_size=1000000,
                                    #    save_path='/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_1m/transaction.pkl'
                                       )

In [ ]:
import pickle as pkl
with open('/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_1m/transaction.pkl', 'rb') as f:
    transaction_list = pkl.load(f)
len(transaction_list)

### Guide to run Apriori algorithm

In [ ]:
from algorithms.apriori import Apriori
import pickle as pkl

# with open('/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_1m/transaction.pkl', 'rb') as f:
#     transaction_list = pkl.load(f)

apriori = Apriori(min_support=0.2, min_confidence=0.5)
apriori.fit(transaction_list)

In [ ]:
# apriori.frequent_itemsets
map_movie_id_to_title(apriori.frequent_itemsets)

### Guide to run Hashtree Apriori algorithm

In [ ]:
from algorithms.hashtree_apriori import HashTreeApriori

hash_tree_apriori = HashTreeApriori(min_support=0.2, min_confidence=0.5, max_leaf_size=64, max_depth=4)
hash_tree_apriori.fit(transaction_list)

In [ ]:
# hash_tree_apriori.frequent_itemsets
map_movie_id_to_title(hash_tree_apriori.frequent_itemsets)

### Guide to run FP-Growth algorithm

In [ ]:
from algorithms.fp_growth import FPGrowth
# Same interface as Apriori
fp_growth = FPGrowth(min_support=0.2, min_confidence=0.5)
fp_growth.fit(transaction_list)

# Get results
fp_growth.get_frequent_itemsets

# Save/load results
# fp_growth.save_results('fp_growth_results.pkl')

In [ ]:
fp_growth.frequent_itemsets
map_movie_id_to_title(fp_growth.frequent_itemsets)

### Guide to run Content-based filtering algorithm (quite computation-exhaustive)

#### Embedding movies & users using multiprocessing
We can use create_user_embedding method in utils.create_embedding to create user embeddings, but that implementation does not support multi-processing.

In [1]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import multiprocessing as mp # For multiprocessing

# Worker function for multiprocessing. Must be defined at the top level.
def process_user_embedding_task(user_id, user_specific_ratings_df, all_movie_embeddings, embedding_s):
    """Calculates embedding for a single user."""
    user_emb = np.zeros(embedding_s)
    denom = 0
    for _, rated_row in user_specific_ratings_df.iterrows():
        movie_id = rated_row['movieId']
        actual_rating = rated_row['rating']

        # IMPORTANT: Check if movie_id exists in movie_embeddings
        if movie_id in all_movie_embeddings:
            current_movie_embedding = all_movie_embeddings[movie_id]
            # Ensure it's a numpy array and has the expected shape
            if isinstance(current_movie_embedding, np.ndarray) and current_movie_embedding.shape == embedding_s:
                if np.any(current_movie_embedding): # Check if embedding is not all zeros
                    denom += 1
                    user_emb += actual_rating * current_movie_embedding
            # else: you might want to log if an embedding is malformed (e.g., wrong type or shape)
        # else: you might want to log if a movie_id from ratings is not in movie_embeddings

    if denom > 0:
        user_emb /= denom
    return user_id, user_emb

def create_user_embedding(ratings_file_path: str = 'data/movielens_20m/rating.csv', 
                          movie_embedding_path: str = 'data/movielens_20m/movie_embedding.pkl',
                          save_path: str = 'data/movielens_20m/user_embedding.pkl'):
    
    try:
        rating = pd.read_csv(ratings_file_path)
        # movies_df = pd.read_csv(movie_file_path) # Not directly used in current embedding logic
    except FileNotFoundError as e:
        print(f"Error: Could not read ratings file. {e}")
        return {}
    
    try:
        with open(movie_embedding_path, 'rb') as f:
            movie_embeddings = pickle.load(f)
    except FileNotFoundError:
        print(f"Error: Movie embedding file not found at {movie_embedding_path}")
        return {}
    except Exception as e:
        print(f"Error loading movie_embeddings: {e}")
        return {}
        
    user_embedding = {}
    
    # Robustly determine embedding shape
    _embedding_shape = None
    if not movie_embeddings or not isinstance(movie_embeddings, dict) or len(movie_embeddings) == 0:
        print("Error: movie_embeddings is empty, not a dictionary, or invalid. Cannot proceed.")
        return {}
    
    # Try to get shape from the first valid NumPy array embedding
    for key in movie_embeddings: # Iterate to find the first valid embedding
        if isinstance(movie_embeddings[key], np.ndarray):
            _embedding_shape = movie_embeddings[key].shape
            break 
    
    if _embedding_shape is None:
        print("Error: Could not determine a valid embedding shape from movie_embeddings (no NumPy arrays found as values).")
        return {}
    
    print(f"Determined embedding shape: {_embedding_shape}")

    # Prepare arguments for each task for multiprocessing
    tasks_args = []
    # Grouping by userId is efficient as each worker gets only the data it needs for one user
    grouped_ratings = rating.groupby('userId')
    for user_id_val, group_df in grouped_ratings:
        tasks_args.append((user_id_val, group_df, movie_embeddings, _embedding_shape))

    if not tasks_args:
        print("No users found in ratings data to process.")
        return {}

    # Determine number of processes
    # Using mp.cpu_count() can be aggressive; mp.cpu_count() // 2 or mp.cpu_count() - 1 is often a good start
    num_processes = max(1, mp.cpu_count() // 2) 
    print(f"Starting user embedding computation with {num_processes} processes for {len(tasks_args)} users...")

    results = []
    # Create a pool of worker processes
    # The `if __name__ == "__main__":` guard is crucial for multiprocessing on some OS (like Windows)
    # when running scripts. For Jupyter notebooks, it's usually not needed for the pool itself,
    # but the worker function must be defined at the top-level.
    try:
        with mp.Pool(processes=num_processes) as pool:
            # Use tqdm for progress bar. pool.starmap executes tasks and returns results in order.
            results = list(tqdm(pool.starmap(process_user_embedding_task, tasks_args), total=len(tasks_args), desc="Processing users"))
    except Exception as e:
        print(f"An error occurred during multiprocessing: {e}")
        print("Consider if the worker function or data being passed is picklable.")
        # Optionally, you could add a fallback to single-threaded processing here for debugging:
        # print("Falling back to single-threaded processing due to error.")
        # results = []
        # for args_tuple in tqdm(tasks_args, desc="Single-threaded fallback"):
        #     results.append(process_user_embedding_task(*args_tuple))


    for res_user_id, res_user_emb in results:
        user_embedding[res_user_id] = res_user_emb

    print(f"Finished processing embeddings for {len(user_embedding)} users.")
    
    if save_path:
        try:
            with open(save_path, 'wb') as f:
                pickle.dump(user_embedding, f)
                print(f"User embeddings saved to {save_path}")
        except Exception as e:
            print(f"Error saving user embeddings to {save_path}: {e}")
    
    return user_embedding


In [ ]:
import os
import sys
from algorithms.content_based_filtering import ContentBasedFiltering
create_embeddings_kwargs = {
    'movies_file_path': 'data/movielens_20m/movies_processed.csv',
    'model_name': 'all-MiniLM-L6-v2',
    'item_embedding_save_path': 'data/movielens_20m/movie_embedding.pkl',
    'ratings_file_path': 'data/movielens_20m/ratings.csv',
    'movie_embedding_path': 'data/movielens_20m/movie_embedding.pkl',
    'user_embedding_save_path': 'data/movielens_20m/user_embedding.pkl'
}
content_based_filtering = ContentBasedFiltering(**create_embeddings_kwargs)

In [ ]:
import pickle as pkl
with open('/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/movie_embedding.pkl', 'wb') as f:
    pkl.dump(content_based_filtering.item_embedding_map, f)

In [ ]:
user_embeddings_path = 'data/movielens_20m/user_embedding.pkl'
user_embeddings = create_user_embedding(ratings_file_path='data/movielens_20m/ratings.csv', 
                                        movie_embedding_path='data/movielens_20m/movie_embedding.pkl',
                                        save_path=user_embeddings_path)

In [ ]:
len(content_based_filtering.item_embedding)

In [ ]:
len(user_embeddings)

In [ ]:
with open('/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/user_embedding.pkl', 'wb') as f:
    pkl.dump(user_embeddings, f)
len(user_embeddings)

In [ ]:
import os
import sys
from algorithms.content_based_filtering import ContentBasedFiltering
create_embeddings_kwargs = {
    'movies_file_path': 'data/movielens_20m/movies_processed.csv',
    'model_name': 'all-MiniLM-L6-v2',
    'item_embedding_save_path': 'data/movielens_20m/movie_embedding.pkl',
    'ratings_file_path': 'data/movielens_20m/ratings.csv',
    'movie_embedding_path': 'data/movielens_20m/movie_embedding.pkl',
    'user_embedding_save_path': 'data/movielens_20m/user_embedding.pkl'
}
content_based_filtering = ContentBasedFiltering(user_embedding_path='data/movielens_20m/user_embedding.pkl', 
                                                item_embedding_path='data/movielens_20m/movie_embedding.pkl',
                                                user_item_interaction_path='data/movielens_20m/user_movies.pkl',
                                                **create_embeddings_kwargs)

In [ ]:
content_based_filtering.item_embedding

In [ ]:
from utils.metrics import cosine_similarity
similarity_matrix = cosine_similarity(content_based_filtering.user_embedding[1].reshape(1, -1), content_based_filtering.item_embedding)

In [7]:
import numpy as np
top_items = np.argsort(similarity_matrix)[0][::-1]

In [12]:
content_based_filtering.recommend(user_id=1, top_k=50)

/home/hinhnv/Hai/KDLVKP/data_mining_code/utils/metrics.py:14: RuntimeWarning: invalid value encountered in divide
  return a @ b.T / (np.linalg.norm(a, axis=1, keepdims=True) * np.linalg.norm(b, axis=1, keepdims=True).T)


[104039,
 129883,
 102090,
 81138,
 49183,
 890,
 126422,
 25921,
 81182,
 48098,
 126432,
 81393,
 126434,
 111287,
 126436,
 107230,
 69670,
 99068,
 48851,
 119677,
 71376,
 79912,
 71343,
 26162,
 128592,
 110457,
 110510,
 107673,
 80233,
 81085,
 103825,
 80557,
 107610,
 117180,
 80810,
 117330,
 80980,
 110641,
 92122,
 110643,
 5795,
 117576,
 82759,
 126579,
 9003,
 68244,
 128149,
 111745,
 106848,
 91697]

In [15]:
import pandas as pd

def get_top_rated_movies_for_user(user_id, top_n=10):
    """
    Retrieve the top N movies rated by a specific user along with movie information.
    
    Parameters:
    -----------
    user_id : int
        The ID of the user
    top_n : int, optional
        Number of top rated movies to return (default: 10)
        
    Returns:
    --------
    DataFrame containing the user's top rated movies with movie information
    """
    global ratings_df, movies_df
    

    user_ratings = ratings_df[ratings_df['userId'] == user_id]
    user_ratings = user_ratings.sort_values(by='rating', ascending=False)
    
    top_rated = user_ratings.head(top_n)

    result = pd.merge(top_rated, movies_df, on='movieId')
    result = result[['movieId', 'title', 'genres', 'rating']]
    
    return result

In [18]:
import pandas as pd
movies_file_path = 'data/movielens_20m/movies_processed.csv'

movies = pd.read_csv(movies_file_path)
movie_genome_tags = movies['genome_tags'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
movie_user_tags = movies['user_tags'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
genres = movies['genres'].apply(lambda x: x.split('|') if isinstance(x, str) else [])

def get_unique_tags(tags_list):
    unique_tags = set()
    for tags in tags_list:
        unique_tags.update([tag for tag in tags])
    return unique_tags

unique_genome_tags = get_unique_tags(movie_genome_tags)
unique_user_tags = get_unique_tags(movie_user_tags)
unique_genres = get_unique_tags(genres)

unique_tags = list(unique_genome_tags.union(unique_genres))

In [26]:
from collections import defaultdict
import pickle
import os

def extract_user_movies(ratings_file_path, output_pickle_path=None, rating_threshold=3.0, chunk_size=1000000):
    """
    Optimized version of extract_user_movies with significant performance improvements.
    
    Key optimizations:
    1. Larger chunk size for better I/O efficiency
    2. Vectorized operations instead of iterrows()
    3. More efficient data structures
    4. Reduced memory allocations
    5. Optional parallel processing
    """
    
    print(f"Processing ratings from: {ratings_file_path}")
    
    # Use regular dict with list comprehension for better performance
    user_movies = {}
    
    if not os.path.exists(ratings_file_path):
        raise FileNotFoundError(f"Ratings file not found: {ratings_file_path}")
    
    try:
        # Read file info first to estimate progress
        total_lines = sum(1 for _ in open(ratings_file_path)) - 1  # Subtract header
        print(f"Total ratings to process: {total_lines:,}")
        
        chunk_iter = pd.read_csv(ratings_file_path, chunksize=chunk_size)
        total_ratings = 0
        chunk_count = 0
        
        for chunk in chunk_iter:
            chunk_count += 1
            print(f"Processing chunk {chunk_count} ({total_ratings:,}/{total_lines:,} ratings)...")
            
            # Apply rating threshold using vectorized operation
            if rating_threshold is not None:
                chunk = chunk[chunk['rating'] >= rating_threshold]
            
            # OPTIMIZATION 1: Use vectorized groupby instead of iterrows()
            # This is much faster than iterating row by row
            grouped = chunk.groupby('userId')['movieId'].apply(list).to_dict()
            
            # OPTIMIZATION 2: Merge dictionaries efficiently
            for user_id, movie_list in grouped.items():
                if user_id in user_movies:
                    # Use set operations for faster merging and deduplication
                    user_movies[user_id].update(movie_list)
                else:
                    user_movies[user_id] = set(movie_list)
            
            total_ratings += len(chunk)
            
            # Progress indicator
            progress = (total_ratings / total_lines) * 100
            print(f"  - Progress: {progress:.1f}% ({len(chunk):,} ratings in this chunk)")
    
    except Exception as e:
        print(f"Error reading file: {e}")
        return {}
    
    # OPTIMIZATION 3: Convert sets to lists in one go
    user_movies_dict = {user_id: list(movie_set) for user_id, movie_set in user_movies.items()}
    
    print(f"\nExtraction Summary:")
    print(f"- Total users: {len(user_movies_dict):,}")
    print(f"- Total ratings processed: {total_ratings:,}")
    if user_movies_dict:
        avg_movies = sum(len(movies) for movies in user_movies_dict.values()) / len(user_movies_dict)
        print(f"- Average movies per user: {avg_movies:.2f}")
    
    # Generate output path if not provided
    if output_pickle_path is None:
        base_name = os.path.splitext(os.path.basename(ratings_file_path))[0]
        output_dir = os.path.dirname(ratings_file_path)
        threshold_suffix = f"_threshold_{rating_threshold}" if rating_threshold else ""
        output_pickle_path = os.path.join(output_dir, f"{base_name}_user_movies{threshold_suffix}.pkl")
    
    # OPTIMIZATION 4: Use protocol 4 for faster pickle serialization
    try:
        with open(output_pickle_path, 'wb') as f:
            pickle.dump(user_movies_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"\nUser-movie mappings saved to: {output_pickle_path}")
        print(f"File size: {os.path.getsize(output_pickle_path) / 1024 / 1024:.2f} MB")
    except Exception as e:
        print(f"Error saving pickle file: {e}")
        return user_movies_dict
    
    return user_movies_dict

In [ ]:
user_movies_dict = extract_user_movies('/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/ratings.csv', output_pickle_path='/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/user_movies.pkl')

In [ ]:
a = (1, 2, 3)
print(a[0])

### Colaborative filtering

In [13]:
import numpy as np
import pandas as pd
from utils.preprocess import create_user_item_matrix
from algorithms.colab_filtering import ColabFiltering

# Attempt to load existing data
user_item_matrix = create_user_item_matrix(
    ratings_file='data/movielens_20m/ratings.csv'
)

In [16]:
# Strategies: 'user_based', 'item_based'
# Similarity metrics: 'cosine', 'pearson'
k_neighbors = 5
colab_filter_user_based_cosine = ColabFiltering(strategy='item_based', similarity_metric='cosine', k=k_neighbors)
colab_filter_user_based_cosine.fit(user_item_matrix)

/home/hinhnv/Hai/KDLVKP/data_mining_code/algorithms/colab_filtering.py:86: RuntimeWarning: invalid value encountered in divide
  pred = np.where(denom>0, nom/denom, -1)
/home/hinhnv/Hai/KDLVKP/data_mining_code/algorithms/colab_filtering.py:86: RuntimeWarning: divide by zero encountered in divide
  pred = np.where(denom>0, nom/denom, -1)


In [17]:
colab_filter_user_based_cosine.recommend(user_id=1, top_k=50)

array([1006,  495, 1002, 1590, 2859,   57, 1275, 1106,  989,   65,  436,
        240, 2810, 3195,  235, 2423,  773, 1132, 1049, 3174,  534, 1039,
       1422, 3173, 3172, 1060, 3177,  101, 2010, 2009, 3171, 3169, 2999,
       3164, 2426, 2892, 3041, 2594, 1373, 2612,  757, 1295,    2, 2872,
       2899,  937, 3165, 2601, 1589, 1581])